# Q1: Fuzzy Logic System — Washing Machine Cycle Time
**Inputs:** Dirt Level, Load Size  
**Output:** Cycle Time (minutes)  
**Tool:** scikit-fuzzy

In [ ]:
# Step 1: Install scikit-fuzzy
!pip install scikit-fuzzy

In [3]:
# Step 2: Import Libraries
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

In [5]:
# Step 3: Define Universe of Discourse (input/output ranges)
dirt_level  = ctrl.Antecedent(np.arange(0, 11, 0.5), 'dirt_level')
load_size   = ctrl.Antecedent(np.arange(0, 11, 0.5), 'load_size')
cycle_time  = ctrl.Consequent(np.arange(0, 81, 1), 'cycle_time')     # 0 to 80 minutes

In [6]:
# Step 4: Define Membership Functions

# --- Dirt Level: Low, Medium, High ---
dirt_level['low']    = fuzz.trapmf(dirt_level.universe,  [0, 0, 2, 4])
dirt_level['medium'] = fuzz.trapmf(dirt_level.universe,  [2, 4, 6, 8])
dirt_level['high']   = fuzz.trapmf(dirt_level.universe,  [6, 8, 10, 10])

# --- Load Size: Small, Medium, Large ---
load_size['small']   = fuzz.trapmf(load_size.universe,   [0, 0, 2, 4])
load_size['medium']  = fuzz.trapmf(load_size.universe,   [2, 4, 6, 8])
load_size['large']   = fuzz.trapmf(load_size.universe,   [6, 8, 10, 10])

# --- Cycle Time: Short, Medium, Long ---
cycle_time['short']  = fuzz.trapmf(cycle_time.universe,  [0, 0, 15, 35])
cycle_time['medium'] = fuzz.trapmf(cycle_time.universe,  [25, 40, 60, 75])
cycle_time['long']   = fuzz.trapmf(cycle_time.universe,  [60, 75, 90, 90])

In [ ]:
# Step 5: Plot Membership Functions
fig, axes = plt.subplots(3, 1, figsize=(6, 12))  # vertical layout

# --- Dirt Level ---
for label in dirt_level.terms:
    axes[0].plot(dirt_level.universe, dirt_level[label].mf, label=label)
axes[0].set_title('Dirt Level (Trapezoidal MF)')
axes[0].legend()
axes[0].grid(True)

# --- Load Size ---
for label in load_size.terms:
    axes[1].plot(load_size.universe, load_size[label].mf, label=label)
axes[1].set_title('Load Size (Trapezoidal MF)')
axes[1].legend()
axes[1].grid(True)

# --- Cycle Time ---
for label in cycle_time.terms:
    axes[2].plot(cycle_time.universe, cycle_time[label].mf, label=label)
axes[2].set_title('Cycle Time (Trapezoidal MF)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('mf_trapezoidal_plot.png', dpi=200)
plt.show()

print('Trapezoidal membership plots saved successfully!')

In [ ]:
# Step 6: Define Fuzzy Rules (at least 6)
rule1 = ctrl.Rule(dirt_level['low'] & load_size['small'], cycle_time['short'])

rule2 = ctrl.Rule(dirt_level['low'] & load_size['large'], cycle_time['medium'])

rule3 = ctrl.Rule(dirt_level['medium'] & load_size['medium'], cycle_time['medium'])

rule4 = ctrl.Rule(dirt_level['high'] & load_size['small'], cycle_time['medium'])

rule5 = ctrl.Rule(dirt_level['high'] & load_size['large'], cycle_time['long'])

rule6 = ctrl.Rule(dirt_level['medium'] & load_size['large'], cycle_time['long'])

rule_default = ctrl.Rule(dirt_level['medium'] | load_size['medium'],cycle_time['medium'])

print('6 fuzzy rules defined successfully!')

In [ ]:
# Step 7: Build the Fuzzy Control System
rule_set = [rule1, rule2, rule3, rule4, rule5, rule6, rule_default]
washing_system = ctrl.ControlSystem(rule_set)
washing_simulator = ctrl.ControlSystemSimulation(washing_system)

In [ ]:
# Step 8 Sample Inputs to test the system and user inputs also allowed
def compute_cycle_time(dirt, load, description):
    if not (0 <= dirt <= 10):
        print("Invalid Dirt Level! Enter value between 0–10.")
        return
    if not (0 <= load <= 10):
        print("Invalid Load Size! Enter value between 0–10.")
        return

    washing_simulator.input['dirt_level'] = dirt
    washing_simulator.input['load_size']  = load

    washing_simulator.compute()
    output_time = washing_simulator.output['cycle_time']

    print(f"{description:<30} {dirt:>6} {load:>6} {output_time:>18.2f}")


# Test Cases
test_cases = [
    (2, 2,  'Low dirt, Small load'),
    (5, 5,  'Medium dirt, Medium load'),
    (8, 8,  'High dirt, Large load'),
    (3, 7,  'Low dirt, Large load'),
    (7, 3,  'High dirt, Small load'),
]

print(f"{'Scenario':<30} {'Dirt':>6} {'Load':>6} {'Cycle Time (min)':>18}")
print('-' * 63)

for dirt, load, label in test_cases:
    compute_cycle_time(dirt, load, label)


# User Input Section
print("\n")
print("Enter custom values for testing")
print("=" * 32)

print("Dirt Level Guide:")
print("  0-3  → Low")
print("  4-6  → Medium")
print("  7-10 → High")

print("=" * 32)

print("Load Size Guide:")
print("  0-3  → Small")
print("  4-6  → Medium")
print("  7-10 → Large")

print("=" * 32)

dirt = float(input("Enter Dirt Level (0-10): "))
load = float(input("Enter Load Size (0-10): "))

compute_cycle_time(dirt, load, "User Input")


In [ ]:
# Step 9: Rule Viewer
from mpl_toolkits.mplot3d import Axes3D

dirt_vals = np.arange(0, 11, 1)
load_vals = np.arange(0, 11, 1)

D, L = np.meshgrid(dirt_vals, load_vals)

T = np.zeros_like(D, dtype=float)

for i in range(D.shape[0]):
    for j in range(D.shape[1]):

        sim = ctrl.ControlSystemSimulation(washing_system)

        sim.input['dirt_level'] = D[i, j]
        sim.input['load_size']  = L[i, j]

        sim.compute()

        T[i, j] = sim.output[cycle_time.label]

# Plot 3D surface
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(D, L, T, cmap='viridis')

ax.set_xlabel('Dirt Level')
ax.set_ylabel('Load Size')
ax.set_zlabel('Cycle Time (min)')
ax.set_title('3D Surface View of Washing Time')

fig.colorbar(surf)

plt.tight_layout()
plt.show()